# Comparing predicted protein structures

Different prediction methods, or different runs of the same method, can give different 3D models of one protein. This notebook loads those models from PDB files, superimposes them, draws them in one view, and measures the remaining differences.

You will:

- point the notebook at two or more PDB files
- superimpose the structures with a rigid alignment
- view each model in its own color, as a cartoon of the secondary structure plus a thin licorice drawing of the atoms
- compare the models with two RMSD matrices: backbone atoms, and all protein atoms

Run the cells from top to bottom. The cell you edit is the list of file paths.


## 1. Setup

The notebook uses [Biopython](https://biopython.org/) to read structures, NumPy for the alignment, pandas and Matplotlib for the RMSD tables, and [py3Dmol](https://3dmol.csb.pitt.edu/) for the interactive view.

Start Jupyter from this project directory, then run the next cell once. If the packages are already installed, it finishes quietly. You can also install them from a terminal with `pip install -r requirements.txt`. If an import in the following cell fails, restart the kernel and run the cells again.


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
%matplotlib inline

from io import StringIO
from pathlib import Path
import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3Dmol
from Bio.PDB import MMCIFParser, PDBIO, PDBParser, is_aa
from Bio.SeqUtils import seq1
from IPython.display import HTML, display

# Backbone heavy atoms, in the order chemists write a peptide unit.
BACKBONE_ATOMS = ("N", "CA", "C", "O")

# Colors chosen to stay distinct for the most common forms of color vision deficiency.
COLORS = [
    "#0072B2",
    "#E69F00",
    "#009E73",
    "#CC79A7",
    "#56B4E9",
    "#D55E00",
    "#F0E442",
    "#000000",
    "#882255",
    "#44AA99",
]


## 2. Choose the structure files

List one path per structure. Use an absolute path, or a path relative to this project directory. A `structures/` folder is a convenient place to put the files. Every file should describe the **same protein** with the **same residue numbering**. PDB (`.pdb`) and mmCIF (`.cif`) files both work.

`CHAIN_ID` keeps a single chain, for example `"A"`. Leave it as `None` to use every chain. Chain IDs are case-sensitive. Set `CHAIN_ID` when a file contains several chains that reuse residue numbers.

`NAMES` is one short name per file, in the same order. These names are what appear on the RMSD tables, the heatmap, and the 3D view. The summary table still shows the file name, so a long path does not have to fit on the figure.


In [ ]:
PDB_FILES = [
    "AlphaFold3/ranked_1.pdb",
    "ColabFold_default/test_a5e17_unrelaxed_rank_001_alphafold2_ptm_model_3_seed_000.pdb",
    "D-I-TASSER/model1.pdb",
    "ESMfold/model1.pdb",
    "RosettaFold/output.pdb",
]

# Short names for the RMSD figure and the 3D view, in the same order as PDB_FILES.
NAMES = [
    "AlphaFold3",
    "ColabFold",
    "D-I-TASSER",
    "ESMfold",
    "RosettaFold",
]

# Example: CHAIN_ID = "A"
CHAIN_ID = None

## 3. Load the structures and pair the atoms

A PDB file stores each atom as a point in a 3D coordinate system. The unit is the ångström (1 Å = 0.1 nm). The origin and the orientation of that coordinate system are arbitrary: the person who wrote the file chose where to place the molecule. Two models of the same fold can therefore have very different raw coordinates.

Before the shapes can be compared, atoms have to be paired. This notebook pairs an atom with the atom that has the same residue number, insertion code, and atom name. Residue numbers need to refer to the same position in the sequence in every file.

Standard amino acids are kept. Water, ions, and other heteroatoms are left out of the comparison and out of the figure. If a file contains several models, the first model is used.


In [ ]:
def resolve_paths(raw_paths):
    """Return existing paths, or raise an error that says which ones to fix."""
    resolved = []
    missing = []
    for raw in raw_paths:
        path = Path(raw).expanduser()
        if not path.is_file():
            alternative = Path.cwd() / path
            if alternative.is_file():
                path = alternative
        if path.is_file():
            resolved.append(path.resolve())
        else:
            missing.append(str(Path(raw).expanduser()))
    if missing:
        lines = "\n".join(f"  - {item}" for item in missing)
        raise FileNotFoundError(
            "Could not find these files. Edit PDB_FILES so each entry points "
            f"to a structure file:\n{lines}"
        )
    if len(resolved) < 2:
        raise ValueError("Add at least two structure files to PDB_FILES.")
    return resolved


def make_labels(paths, names):
    """Keep the short display names, and make any duplicates unique."""
    if len(names) != len(paths):
        raise ValueError(
            f"NAMES has {len(names)} entries, but PDB_FILES has {len(paths)}. "
            "Add one short name per file, in the same order."
        )
    chosen = [str(name) for name in names]
    seen = {}
    unique = []
    for label in chosen:
        seen[label] = seen.get(label, 0) + 1
        unique.append(label if seen[label] == 1 else f"{label} ({seen[label]})")
    return unique


def is_hydrogen(atom):
    """True for hydrogen or deuterium, including files that omit the element column."""
    element = (atom.element or "").strip().upper()
    if element:
        return element in {"H", "D"}
    name = atom.get_name().strip()
    return name.startswith("H") or (
        len(name) > 1 and name[0].isdigit() and name[1] == "H"
    )


def read_structure(path, label):
    """Load the first model from a PDB or mmCIF file."""
    if path.suffix.lower() == ".gz":
        raise ValueError(
            f"{label}: decompress {path.name} before loading it. "
            "Gzipped structure files are left for you to unpack."
        )
    if path.suffix.lower() in {".cif", ".mmcif"}:
        structure = MMCIFParser(QUIET=True).get_structure(label, str(path))
    else:
        structure = PDBParser(QUIET=True).get_structure(label, str(path))
    n_models = len(list(structure.get_models()))
    if n_models > 1:
        print(f"{label}: this file contains {n_models} models. Using the first model.")
    return structure


def amino_acid_atoms(structure, chain_id, label):
    """Collect one record per atom of each standard amino acid.

    The pairing key is (residue number, insertion code, atom name). The same
    key in two files is treated as the same atom.
    """
    model = next(structure.get_models())
    available = [chain.id for chain in model]
    records = []
    seen = {}
    for chain in model:
        if chain_id is not None and chain.id != chain_id:
            continue
        for residue in chain:
            hetero_flag, resseq, icode = residue.id
            if hetero_flag.strip() or not is_aa(residue, standard=True):
                continue
            for atom in residue.get_atoms():
                name = atom.get_name().strip()
                key = (resseq, icode.strip(), name)
                if key in seen:
                    raise ValueError(
                        f"{label}: residue {resseq}{icode.strip()} atom {name} "
                        f"appears in chains {seen[key]!r} and {chain.id!r}. "
                        "Set CHAIN_ID to the chain you want to compare."
                    )
                seen[key] = chain.id
                records.append(
                    {
                        "key": key,
                        "chain": chain.id,
                        "resseq": resseq,
                        "icode": icode.strip(),
                        "resname": residue.get_resname().strip(),
                        "atom": name,
                        "coord": np.asarray(atom.get_coord(), dtype=float),
                        "hydrogen": is_hydrogen(atom),
                    }
                )
    if not records:
        chain_list = ", ".join(repr(chain) for chain in available) or "(none)"
        where = f" in chain {chain_id!r}" if chain_id is not None else ""
        raise ValueError(
            f"{label}: no standard amino-acid atoms were found{where}. "
            f"Chains in this file: {chain_list}."
        )
    return records


def sequence_of(records):
    """One-letter sequence in residue-number order."""
    residues = []
    seen = set()
    for rec in sorted(records, key=lambda item: (item["resseq"], item["icode"])):
        residue_id = (rec["resseq"], rec["icode"])
        if residue_id in seen:
            continue
        seen.add(residue_id)
        residues.append(rec["resname"])
    letters = []
    for resname in residues:
        try:
            letters.append(seq1(resname))
        except KeyError:
            letters.append("X")
    return "".join(letters)


def shared_keys(models, backbone_only):
    """Atom keys present in every loaded structure."""
    key_sets = []
    for model in models:
        keys = set()
        for rec in model["records"]:
            if backbone_only and rec["atom"] not in BACKBONE_ATOMS:
                continue
            keys.add(rec["key"])
        key_sets.append(keys)
    return sorted(set.intersection(*key_sets))


def coordinates_for(records, keys):
    """Stack paired coordinates in a shared key order. Shape (N, 3)."""
    lookup = {rec["key"]: rec["coord"] for rec in records}
    return np.vstack([lookup[key] for key in keys])


In [ ]:
paths = resolve_paths(PDB_FILES)
labels = make_labels(paths, NAMES)

models = []
for path, label in zip(paths, labels):
    structure = read_structure(path, label)
    records = amino_acid_atoms(structure, CHAIN_ID, label)
    models.append(
        {
            "label": label,
            "path": path,
            "structure": structure,
            "records": records,
            "sequence": sequence_of(records),
        }
    )

rows = []
for model in models:
    records = model["records"]
    residue_ids = sorted({(rec["resseq"], rec["icode"]) for rec in records})
    chains = sorted({rec["chain"] for rec in records})
    chain_text = ", ".join(chain if str(chain).strip() else "(blank)" for chain in chains)
    rows.append(
        {
            "Structure": model["label"],
            "File": model["path"].name,
            "Chains": chain_text,
            "Residues": len(residue_ids),
            "Residue numbers": f"{residue_ids[0][0]}–{residue_ids[-1][0]}",
            "Backbone atoms": sum(rec["atom"] in BACKBONE_ATOMS for rec in records),
            "Heavy atoms": sum(not rec["hydrogen"] for rec in records),
            "Hydrogens": sum(rec["hydrogen"] for rec in records),
        }
    )
display(pd.DataFrame(rows))

sequences = [model["sequence"] for model in models]
if len(set(sequences)) == 1:
    print(f"All files contain the same amino-acid sequence ({len(sequences[0])} residues).")
else:
    print(
        "These files contain different amino-acid sequences. "
        "RMSD still pairs atoms by residue number and atom name."
    )
    for model in models:
        sequence = model["sequence"]
        preview = sequence if len(sequence) <= 70 else sequence[:67] + "..."
        print(f"  {model['label']}: length {len(sequence)}, {preview}")

backbone_keys = shared_keys(models, backbone_only=True)
all_atom_keys = shared_keys(models, backbone_only=False)
if len(backbone_keys) < 3 or len(all_atom_keys) < 3:
    raise ValueError(
        "Fewer than 3 atoms are shared by every file. Check CHAIN_ID, and "
        "check that residue numbers refer to the same residues in every file."
    )

smallest_backbone = min(
    sum(rec["atom"] in BACKBONE_ATOMS for rec in model["records"]) for model in models
)
if len(backbone_keys) < 0.5 * smallest_backbone:
    print(
        "Fewer than half of the backbone atoms could be paired. "
        "The files may use different residue numbering, or CHAIN_ID may be unset "
        "on a multi-chain file."
    )

hydrogen_keys = {
    rec["key"] for model in models for rec in model["records"] if rec["hydrogen"]
}
n_shared_hydrogens = sum(key in hydrogen_keys for key in all_atom_keys)
print(
    f"Backbone comparison uses {len(backbone_keys)} atoms "
    "(N, CA, C, and O) present in every structure."
)
print(
    f"All-atom comparison uses {len(all_atom_keys)} protein atoms present in every structure "
    f"({len(all_atom_keys) - n_shared_hydrogens} heavy atoms, {n_shared_hydrogens} hydrogens)."
)
for model in models:
    own_keys = {rec["key"] for rec in model["records"]}
    dropped = len(own_keys - set(all_atom_keys))
    if dropped:
        print(f"  {model['label']}: {dropped} protein atoms are missing from at least one other file.")

for model in models:
    model["backbone"] = coordinates_for(model["records"], backbone_keys)
    model["all_atom"] = coordinates_for(model["records"], all_atom_keys)


## 4. Structural alignment

The raw coordinates still include an arbitrary placement of each model in space. A **rigid alignment**, also called a superposition, removes that placement. The mobile structure may be rotated and translated. Bond lengths and angles stay as they are, and the structure is not mirrored.

Take paired atoms $\mathbf{x}_i$ in the mobile structure and $\mathbf{y}_i$ in the target. Write $\bar{\mathbf{x}}$ and $\bar{\mathbf{y}}$ for their centroids, the average positions. In this notebook each atom is stored as one row of an array, so the fitted position of a mobile atom is

$$
\mathbf{x}_i' = (\mathbf{x}_i - \bar{\mathbf{x}})\, R + \bar{\mathbf{y}}
$$

Subtracting $\bar{\mathbf{x}}$ and adding $\bar{\mathbf{y}}$ puts the two centroids on top of each other. That is the best translation. The $3 \times 3$ matrix $R$ is the rotation that makes the centered atoms agree as well as possible.

$R$ comes from the Kabsch algorithm. The algorithm forms the $3 \times 3$ covariance matrix of the paired, centered coordinates and computes its singular value decomposition (SVD). A final check keeps $\det R = +1$. That value means $R$ is a rotation. A determinant of $-1$ would mirror the protein. Proteins are chiral, so a mirror image is not a meaningful alignment.

The rotation is estimated from the paired backbone atoms. The same rotation and translation are then applied to every atom, so the side chains move together with the backbone.


In [ ]:
def superimpose(mobile, target):
    """Rotate and translate `mobile` onto `target`.

    Both arrays have shape (N, 3): one paired atom per row, in angstroms.
    Returns the aligned mobile coordinates, R, the two centroids, and the RMSD.
    """
    mobile = np.asarray(mobile, dtype=float)
    target = np.asarray(target, dtype=float)
    if mobile.shape != target.shape or mobile.ndim != 2 or mobile.shape[1] != 3:
        raise ValueError("mobile and target must both have shape (N, 3).")
    if mobile.shape[0] < 3:
        raise ValueError("At least 3 paired atoms are required for an alignment.")

    # Best translation: superimpose the centroids.
    x_bar = mobile.mean(axis=0)
    y_bar = target.mean(axis=0)
    mobile_centered = mobile - x_bar
    target_centered = target - y_bar

    # Kabsch rotation from the SVD of the 3x3 covariance matrix.
    covariance = mobile_centered.T @ target_centered
    left, _, vt = np.linalg.svd(covariance)
    rotation = left @ vt
    # det = -1 would be a reflection. Flip one axis to force a proper rotation.
    if np.linalg.det(rotation) < 0.0:
        vt = vt.copy()
        vt[-1] *= -1.0
        rotation = left @ vt

    aligned = mobile_centered @ rotation + y_bar
    difference = aligned - target
    rmsd = float(np.sqrt(np.mean(np.sum(difference * difference, axis=1))))
    return aligned, rotation, x_bar, y_bar, rmsd


def apply_transform(coords, rotation, x_bar, y_bar):
    """Apply a superposition that was fitted on one atom selection."""
    coords = np.asarray(coords, dtype=float)
    return (coords - x_bar) @ rotation + y_bar


def pairwise_rmsd(coordinate_sets):
    """RMSD after a separate optimal alignment of every pair."""
    n_structures = len(coordinate_sets)
    matrix = np.zeros((n_structures, n_structures))
    for i in range(n_structures):
        for j in range(i + 1, n_structures):
            *_, value = superimpose(coordinate_sets[i], coordinate_sets[j])
            matrix[i, j] = matrix[j, i] = value
    return matrix

## 5. RMSD

Once the arbitrary placement has been removed, the **root-mean-square deviation** is the typical distance that remains between paired atoms:

$$
\mathrm{RMSD} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left\| \mathbf{x}_i' - \mathbf{y}_i \right\|^2}
$$

$N$ is the number of paired atoms. $\mathbf{x}_i'$ is the aligned mobile atom and $\mathbf{y}_i$ is its partner in the other structure. The unit is the ångström. An RMSD of 0 Å means those atoms land on top of each other.

The square gives extra weight to the atoms that are farthest apart. A few displaced loops raise the RMSD more than the same total error spread evenly along the chain.

Each cell of a matrix is the RMSD of one pair after that pair has been aligned on its own. The matrix is symmetric, because aligning A onto B leaves the same distances as aligning B onto A. Every diagonal entry is 0 Å, because a structure matches itself exactly.

The two matrices use the same structures and answer different questions. Each matrix is fitted on its own atoms, so the all-atom RMSD can be smaller than the backbone RMSD for the same pair: a large number of agreeing side-chain atoms can outweigh a few displaced backbone atoms.

- **Backbone RMSD** uses N, Cα, C, and O. It describes the path of the chain.
- **All-atom RMSD** uses every protein atom that is present in all of the files, including side chains. It also rises when rotamers differ and the backbone agrees. Hydrogen atoms are included when every file contains that hydrogen. Many prediction files contain no hydrogens, and then the all-atom matrix is a heavy-atom matrix.

As a rough guide for the backbone of a globular domain: below about 1 Å the traces are very close, about 1–2 Å usually means the same fold with local differences, and above about 3 Å the backbones differ substantially. Long flexible tails (at the N- and C-terminus of the amino acid chain) can raise the RMSD between two models that otherwise feature the same structure in the core of the protein.


In [ ]:
labels = [model["label"] for model in models]
backbone_rmsd = pairwise_rmsd([model["backbone"] for model in models])
all_atom_rmsd = pairwise_rmsd([model["all_atom"] for model in models])

print(
    f"Backbone matrix: N = {len(backbone_keys)} atoms per structure. "
    f"All-atom matrix: N = {len(all_atom_keys)} atoms per structure."
)
print("Both tables and both panels use the same color scale. Darker red means a larger RMSD.")

n_labels = len(labels)
vmax = float(max(backbone_rmsd.max(), all_atom_rmsd.max(), 0.0))
if vmax == 0.0:
    vmax = 1.0
side = max(4.6, 0.48 * n_labels + 2.8)
fig, axes = plt.subplots(1, 2, figsize=(side * 2.05, side), constrained_layout=True)
panels = (
    (axes[0], backbone_rmsd, "Backbone (N, Cα, C, O)"),
    (axes[1], all_atom_rmsd, "All protein atoms"),
)
for axis, matrix, title in panels:
    axis.imshow(matrix, cmap="YlOrRd", vmin=0.0, vmax=vmax)
    axis.set_title(title)
    axis.set_xticks(range(n_labels), labels, rotation=40, ha="right")
    axis.set_yticks(range(n_labels), labels)
    axis.set_xlabel("Structure")
    axis.set_ylabel("Structure")
    if n_labels <= 10:
        fontsize = max(7, 12 - max(0, n_labels - 3))
        for row in range(n_labels):
            for col in range(n_labels):
                value = matrix[row, col]
                ink = "white" if value > 0.62 * vmax else "black"
                axis.text(
                    col,
                    row,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    color=ink,
                    fontsize=fontsize,
                )
colorbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=plt.Normalize(vmin=0.0, vmax=vmax), cmap="YlOrRd"),
    ax=axes,
    fraction=0.046,
    pad=0.04,
)
colorbar.set_label("RMSD (Å)")
fig.suptitle("Pairwise RMSD after optimal superposition")
plt.show()


## 6. View the superimposed structures

The figure places every model on top of the first file. The fit uses the shared backbone atoms, and that same motion is applied to the whole model. Scroll to zoom and drag to rotate.

The cartoon follows the secondary structure. The thin sticks are a licorice representation: every protein atom is drawn with the same small radius. Each model has its own color, listed above the viewer.

This picture and the RMSD matrices are related but not identical. Here, every model is aligned to the first file. In the matrices, each pair gets its own optimal alignment, so each cell reports the smallest RMSD for that pair.


In [ ]:
def protein_pdb_text(structure, chain_id, rotation, x_bar, y_bar):
    """PDB text for the amino acids after a rigid motion has been applied."""
    moved = copy.deepcopy(structure)
    extra_models = list(moved.get_models())[1:]
    for extra in extra_models:
        moved.detach_child(extra.id)
    for chain in list(next(moved.get_models())):
        if chain_id is not None and chain.id != chain_id:
            chain.parent.detach_child(chain.id)
            continue
        for residue in list(chain):
            hetero_flag = residue.id[0]
            if hetero_flag.strip() or not is_aa(residue, standard=True):
                chain.detach_child(residue.id)
                continue
            for atom in residue.get_atoms():
                atom.set_coord(
                    apply_transform(atom.get_coord(), rotation, x_bar, y_bar)
                )
    buffer = StringIO()
    writer = PDBIO()
    writer.set_structure(moved)
    writer.save(buffer)
    return buffer.getvalue()


colors = [COLORS[index % len(COLORS)] for index in range(len(models))]
reference_backbone = models[0]["backbone"]
pdb_texts = []
for index, model in enumerate(models):
    if index == 0:
        rotation = np.eye(3)
        x_bar = np.zeros(3)
        y_bar = np.zeros(3)
    else:
        _, rotation, x_bar, y_bar, _ = superimpose(
            model["backbone"], reference_backbone
        )
    pdb_texts.append(
        protein_pdb_text(model["structure"], CHAIN_ID, rotation, x_bar, y_bar)
    )

print(
    f"Aligned every model onto {models[0]['label']} "
    f"using {len(backbone_keys)} shared backbone atoms."
)

chips = []
for label, color in zip(labels, colors):
    chips.append(
        "<span style='display:inline-block;margin:0 18px 8px 0;font-family:sans-serif;font-size:15px;'>"
        f"<span style='display:inline-block;width:14px;height:14px;background:{color};"
        "border:1px solid #333;vertical-align:middle;margin-right:6px;'></span>"
        f"{label}</span>"
    )
display(HTML("<div style='margin:6px 0 10px;'>" + "".join(chips) + "</div>"))

view = py3Dmol.view(width=920, height=620)
for pdb_text in pdb_texts:
    view.addModel(pdb_text, "pdb")
for index, color in enumerate(colors):
    view.setStyle(
        {"model": index},
        {
            "cartoon": {"color": color},
            "stick": {"color": color, "radius": 0.1},
        },
    )
view.setBackgroundColor("white")
view.zoomTo()
view.show()